# Parking Sign Detection - YOLOv8 Training

Training a parking sign detector for North American street signs using YOLOv8.

**Dataset:** 1,300 images, 6 classes (A, P, PX, R, S, Temp)

**Goal:** Find optimal augmentation strategy for best detection accuracy.

## 1. Setup

In [ ]:
!pip install ultralytics -q

In [ ]:
import os
from pathlib import Path
import json

from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt

# Kaggle dataset path (update after uploading)
DATASET_PATH = Path("/kaggle/input/parking-sign-coco")
OUTPUT_PATH = Path("/kaggle/working")

print(f"Dataset exists: {DATASET_PATH.exists()}")
if DATASET_PATH.exists():
    print(f"Contents: {list(DATASET_PATH.iterdir())}")

## 2. Convert COCO to YOLO Format

In [ ]:
def convert_coco_to_yolo(coco_json_path: Path, output_dir: Path):
    """Convert COCO JSON annotations to YOLO txt format."""
    with open(coco_json_path) as f:
        coco = json.load(f)

    images = {img["id"]: img for img in coco["images"]}
    labels_dir = output_dir / "labels"
    labels_dir.mkdir(parents=True, exist_ok=True)

    img_annotations = {}
    for ann in coco["annotations"]:
        img_id = ann["image_id"]
        if img_id not in img_annotations:
            img_annotations[img_id] = []
        img_annotations[img_id].append(ann)

    for img_id, img_info in images.items():
        img_w, img_h = img_info["width"], img_info["height"]
        stem = Path(img_info["file_name"]).stem
        label_file = labels_dir / f"{stem}.txt"

        lines = []
        for ann in img_annotations.get(img_id, []):
            x, y, w, h = ann["bbox"]
            x_center = (x + w / 2) / img_w
            y_center = (y + h / 2) / img_h
            w_norm, h_norm = w / img_w, h / img_h
            
            class_id = ann["category_id"]
            if class_id == 0:  # Skip parent "parking" category
                continue
            class_id -= 1
            lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")

        label_file.write_text("\n".join(lines))

    print(f"Converted {len(images)} images")


# Run conversion
WORK_DATASET = OUTPUT_PATH / "dataset"

for split in ["train", "valid", "test"]:
    # Copy images
    src_img = DATASET_PATH / split
    dst_dir = WORK_DATASET / split / "images"
    dst_dir.mkdir(parents=True, exist_ok=True)
    
    # Symlink images to avoid copying
    for img in src_img.glob("*.jpg"):
        (dst_dir / img.name).symlink_to(img)
    
    # Convert annotations
    coco_json = DATASET_PATH / split / "_annotations.coco.json"
    if coco_json.exists():
        convert_coco_to_yolo(coco_json, WORK_DATASET / split)

print(f"\nDataset prepared at {WORK_DATASET}")

In [ ]:
# Create data.yaml
DATA_YAML = WORK_DATASET / "data.yaml"

yaml_content = f"""# Parking Sign Detection Dataset
path: {WORK_DATASET}
train: train/images
val: valid/images
test: test/images

nc: 6
names: ['A', 'P', 'PX', 'R', 'S', 'Temp']
"""

DATA_YAML.write_text(yaml_content)
print(f"Created {DATA_YAML}")
print(yaml_content)

## 3. Augmentation Experiments

### Pre-applied augmentations (Roboflow):
- 3x augmented versions per source image
- Rotation: ±15°
- Brightness: ±15%
- Gaussian blur: 0-2.5px

### Additional YOLOv8 augmentations to test:
- Mosaic: Combines 4 images into one
- MixUp: Blends two images
- HSV: Color space augmentation
- Geometric: Scale, translate, perspective

In [ ]:
# Common training parameters
BASE_PARAMS = {
    "data": str(DATA_YAML),
    "epochs": 100,
    "imgsz": 512,
    "batch": 16,
    "patience": 20,
    "save_period": 25,
    "project": str(OUTPUT_PATH / "runs"),
    "exist_ok": True,
    "pretrained": True,
    "verbose": True,
}

# Experiment configurations
EXPERIMENTS = {
    "exp1_baseline": {
        # Minimal augmentation - test Roboflow preprocessing alone
        "mosaic": 0.0,
        "mixup": 0.0,
        "copy_paste": 0.0,
        "degrees": 0.0,
        "translate": 0.1,
        "scale": 0.0,
        "fliplr": 0.0,
        "flipud": 0.0,
    },
    "exp2_mosaic": {
        # Add mosaic augmentation
        "mosaic": 1.0,
        "mixup": 0.0,
        "copy_paste": 0.0,
        "degrees": 0.0,
        "translate": 0.1,
        "scale": 0.5,
        "fliplr": 0.5,
        "flipud": 0.0,
    },
    "exp3_hsv": {
        # Add HSV color augmentation
        "mosaic": 1.0,
        "hsv_h": 0.015,
        "hsv_s": 0.7,
        "hsv_v": 0.4,
        "translate": 0.1,
        "scale": 0.5,
        "fliplr": 0.5,
    },
    "exp4_geometric": {
        # Add geometric transformations
        "mosaic": 1.0,
        "degrees": 15.0,
        "translate": 0.1,
        "scale": 0.5,
        "shear": 5.0,
        "perspective": 0.0005,
        "fliplr": 0.5,
    },
    "exp5_full": {
        # All augmentations combined
        "mosaic": 1.0,
        "mixup": 0.1,
        "copy_paste": 0.1,
        "hsv_h": 0.015,
        "hsv_s": 0.7,
        "hsv_v": 0.4,
        "degrees": 15.0,
        "translate": 0.1,
        "scale": 0.5,
        "shear": 5.0,
        "perspective": 0.0005,
        "fliplr": 0.5,
        "flipud": 0.0,
    },
}

print(f"Configured {len(EXPERIMENTS)} experiments")

## 4. Run Training Experiments

In [ ]:
results = {}

for exp_name, aug_params in EXPERIMENTS.items():
    print(f"\n{'='*60}")
    print(f"Running {exp_name}")
    print(f"{'='*60}\n")
    
    model = YOLO("yolov8n.pt")
    
    train_params = {**BASE_PARAMS, **aug_params, "name": exp_name}
    train_results = model.train(**train_params)
    
    # Store results
    results[exp_name] = {
        "mAP50": train_results.results_dict.get("metrics/mAP50(B)", 0),
        "mAP50-95": train_results.results_dict.get("metrics/mAP50-95(B)", 0),
        "precision": train_results.results_dict.get("metrics/precision(B)", 0),
        "recall": train_results.results_dict.get("metrics/recall(B)", 0),
    }
    
    print(f"\n{exp_name} Results:")
    for metric, value in results[exp_name].items():
        print(f"  {metric}: {value:.4f}")

## 5. Compare Results

In [ ]:
# Create comparison DataFrame
df = pd.DataFrame(results).T
df = df.sort_values("mAP50-95", ascending=False)

print("\n" + "="*60)
print("EXPERIMENT RESULTS COMPARISON")
print("="*60)
print(df.to_string())

# Save to CSV
df.to_csv(OUTPUT_PATH / "experiment_results.csv")
print(f"\nResults saved to {OUTPUT_PATH / 'experiment_results.csv'}")

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# mAP comparison
ax1 = axes[0]
x = range(len(df))
width = 0.35
ax1.bar([i - width/2 for i in x], df["mAP50"], width, label="mAP50", color="steelblue")
ax1.bar([i + width/2 for i in x], df["mAP50-95"], width, label="mAP50-95", color="coral")
ax1.set_xticks(x)
ax1.set_xticklabels(df.index, rotation=45, ha="right")
ax1.set_ylabel("Score")
ax1.set_title("mAP Comparison")
ax1.legend()
ax1.set_ylim(0, 1)

# Precision/Recall comparison
ax2 = axes[1]
ax2.bar([i - width/2 for i in x], df["precision"], width, label="Precision", color="forestgreen")
ax2.bar([i + width/2 for i in x], df["recall"], width, label="Recall", color="darkorange")
ax2.set_xticks(x)
ax2.set_xticklabels(df.index, rotation=45, ha="right")
ax2.set_ylabel("Score")
ax2.set_title("Precision/Recall Comparison")
ax2.legend()
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(OUTPUT_PATH / "experiment_comparison.png", dpi=150)
plt.show()

print(f"\nChart saved to {OUTPUT_PATH / 'experiment_comparison.png'}")

## 6. Evaluate Best Model

In [ ]:
# Find best experiment
best_exp = df["mAP50-95"].idxmax()
print(f"Best experiment: {best_exp}")
print(f"mAP50-95: {df.loc[best_exp, 'mAP50-95']:.4f}")

# Load best model
best_model_path = OUTPUT_PATH / "runs" / best_exp / "weights" / "best.pt"
best_model = YOLO(best_model_path)

# Run validation on test set
print("\nEvaluating on test set...")
test_results = best_model.val(data=str(DATA_YAML), split="test")

print(f"\nTest Set Results:")
print(f"  mAP50: {test_results.results_dict.get('metrics/mAP50(B)', 0):.4f}")
print(f"  mAP50-95: {test_results.results_dict.get('metrics/mAP50-95(B)', 0):.4f}")

In [ ]:
# Show confusion matrix
from IPython.display import Image, display

confusion_matrix_path = OUTPUT_PATH / "runs" / best_exp / "confusion_matrix.png"
if confusion_matrix_path.exists():
    display(Image(filename=str(confusion_matrix_path)))

## 7. Export Best Model

In [ ]:
# Export to ONNX for deployment
print("Exporting best model to ONNX...")
onnx_path = best_model.export(format="onnx", imgsz=512, simplify=True)
print(f"Exported to: {onnx_path}")

# Also save PyTorch format
import shutil
final_model_path = OUTPUT_PATH / "parking_sign_detector.pt"
shutil.copy(best_model_path, final_model_path)
print(f"PyTorch model: {final_model_path}")

## 8. Conclusions

### Dataset
- 1,300 images with 6 parking sign classes
- Pre-augmented 3x by Roboflow (rotation, brightness, blur)
- 512x512 resolution

### Experiments
1. **Baseline**: Roboflow augmentation only
2. **Mosaic**: Added mosaic augmentation
3. **HSV**: Added color space augmentation
4. **Geometric**: Added perspective transforms
5. **Full**: All augmentations combined

### Key Findings
*[Fill in after running experiments]*

- Best augmentation strategy: ...
- Final mAP50-95: ...
- Training time: ...

### Model Download
- PyTorch: `parking_sign_detector.pt`
- ONNX: `parking_sign_detector.onnx`

In [ ]:
# Print final summary
print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"\nBest Model: {best_exp}")
print(f"mAP50-95: {df.loc[best_exp, 'mAP50-95']:.4f}")
print(f"\nOutput files:")
print(f"  - {OUTPUT_PATH / 'parking_sign_detector.pt'}")
print(f"  - {OUTPUT_PATH / 'experiment_results.csv'}")
print(f"  - {OUTPUT_PATH / 'experiment_comparison.png'}")